# Pydantic AI Agent Harness — Tutorial Notebook

This notebook is a self-contained walkthrough of what
[`pydantic_agent.py`](./pydantic_agent.py) does when it runs an agentic AI job.

**What you will see:**
1. Editable agent instructions (`INSTRUCTION_MD`) and input payloads (`PAYLOAD_1` … `PAYLOAD_3`)
2. Scoped tools the agent uses to read input and write output
3. System prompt assembly
4. A single **OpenAI-compatible** model client (`OPENAI_BASE_URL` + `OPENAI_API_KEY` + `MODEL`)
5. The agentic loop — tool calls, results, and streamed assistant text

**One API shape, many backends** — anything with an OpenAI-compatible `/v1` API works:

| Backend | `OPENAI_BASE_URL` | `OPENAI_API_KEY` | Example `MODEL` |
|---------|-------------------|------------------|-----------------|
| Ollama (local) | `http://localhost:11434/v1` | `local` (any placeholder) | `llama3.2` |
| vLLM / LM Studio | `http://localhost:8000/v1` | your token if required | `mistralai/Mistral-7B-Instruct-v0.3` |
| [OpenRouter](https://openrouter.ai) | `https://openrouter.ai/api/v1` | `sk-or-...` | `openai/gpt-5.4-nano` |

OpenRouter exposes the same Chat Completions API — point `OPENAI_BASE_URL` at OpenRouter and use your OpenRouter key as `OPENAI_API_KEY`.

Run cells top-to-bottom. Configure secrets in **Step 1** before running the agent.


## Architecture

```
Notebook cells  →  run_agent()
                       |  build_model()  →  OpenAIChatModel(OPENAI_BASE_URL)
                       |  make_tools()   →  scoped Python functions
                       v
                 pydantic_ai.Agent
                       |  agent.run_stream_events()
                       v
         FunctionToolCallEvent / FunctionToolResultEvent / PartDeltaEvent
                       →  printed log lines
```

The agent interacts with the world **only** through these tools:
`list_input_files`, `read_input_file`, `write_output`, `append_output`,
`list_output_files`, `run_command`, and `web_fetch`.


## Step 0 — Install dependencies

Uncomment and run if needed:

In [ ]:
%pip install -q pydantic-ai openai httpx ipython


## Step 1 — Configuration

All model backends use the same three settings: **`OPENAI_BASE_URL`**, **`OPENAI_API_KEY`**, and **`MODEL`**.

### Required variables

| Variable | Purpose | Example |
|----------|---------|---------|
| `OPENAI_API_KEY` | Bearer token for the API at `OPENAI_BASE_URL` | `sk-or-...` (OpenRouter) or `local` (Ollama) |
| `OPENAI_BASE_URL` | OpenAI-compatible API root | `https://openrouter.ai/api/v1` |
| `MODEL` | Model ID from that server's `/v1/models` | `anthropic/claude-haiku-4.5` |

Optional: `MAX_TURNS`, `MAX_OUTPUT_TOKENS`, `ALLOW_SHELL`, `SHELL_TIMEOUT`.

The code cell loads values from **Google Colab secrets**, **OpenShift AI Workbench env vars**, or your **local shell** — never hardcode keys in the notebook.

---

### Option A — Google Colab

1. Click the **key icon** (Secrets) in the left sidebar.
2. Add secrets (enable **Notebook access** for each):
   - `OPENAI_API_KEY` → your API key (`sk-or-...` for OpenRouter)
   - `OPENAI_BASE_URL` → e.g. `https://openrouter.ai/api/v1`
   - `MODEL` → e.g. `anthropic/claude-haiku-4.5`
3. The configuration cell reads them via `userdata.get('OPENAI_API_KEY')`.

---

### Option B — OpenShift AI Workbench

1. Create a Secret in your Data Science Project:

```bash
oc create secret generic pydantic-agent-config \
  --from-literal=OPENAI_API_KEY='sk-or-...' \
  --from-literal=OPENAI_BASE_URL='https://openrouter.ai/api/v1' \
  --from-literal=MODEL='anthropic/claude-haiku-4.5' \
  -n <your-project>
```

2. In **Workbenches** → create or edit workbench → **Environment variables** → **Add variable**:
   - Name: `OPENAI_API_KEY` · Type: **Secret** · Secret: `pydantic-agent-config` · Key: `OPENAI_API_KEY`
   - Repeat for `OPENAI_BASE_URL` and `MODEL`.
3. **Restart the workbench** after changes. The notebook reads `os.environ[...]`.

For an in-cluster model server, set `OPENAI_BASE_URL` to its service URL
(e.g. `http://llama-service.<namespace>.svc.cluster.local:8000/v1`).

---

### Option C — Local Jupyter / VS Code

```bash
export OPENAI_BASE_URL=https://openrouter.ai/api/v1
export OPENAI_API_KEY=sk-or-...
export MODEL=anthropic/claude-haiku-4.5
```


In [ ]:
# Depending on the above options used, set the required variables as required.

In [ ]:
import os
from pathlib import Path


def get_secret(name: str, default: str = "") -> str:
    """Colab userdata → environment variable → mounted secret file."""
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name)
    except Exception:
        pass

    value = os.environ.get(name, "")
    if value:
        return value

    secret_mount_dir = os.environ.get("SECRET_MOUNT_DIR", "/run/secrets")
    secret_file = Path(secret_mount_dir) / name
    if secret_file.is_file():
        return secret_file.read_text(encoding="utf-8").strip()

    return default


def get_config(name: str, default: str = "") -> str:
    return os.environ.get(name) or get_secret(name, default)


# --- OpenAI-compatible endpoint (Ollama, vLLM, OpenRouter, etc.) --------------
OPENAI_BASE_URL = get_config("OPENAI_BASE_URL", "http://localhost:11434/v1").rstrip("/")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "local")
MODEL = get_config("MODEL", "llama3.2")

MAX_TURNS = int(get_config("MAX_TURNS", "50"))
MAX_OUTPUT_TOKENS = int(get_config("MAX_OUTPUT_TOKENS", "16384"))
ALLOW_SHELL = get_config("ALLOW_SHELL", "true").lower() not in ("0", "false", "no", "off")
SHELL_TIMEOUT = int(get_config("SHELL_TIMEOUT", "60"))

WORK_DIR = Path("notebook_workspace")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"

print(f"OPENAI_BASE_URL : {OPENAI_BASE_URL}")
print(f"MODEL           : {MODEL}")
print(f"OPENAI_API_KEY set: {bool(OPENAI_API_KEY and OPENAI_API_KEY != 'local')}")
if OPENAI_BASE_URL.endswith("openrouter.ai/api/v1") and not OPENAI_API_KEY.startswith("sk-or-"):
    print("WARNING: OpenRouter usually requires OPENAI_API_KEY starting with sk-or-")


## Step 2 — Agent instructions

Edit `INSTRUCTION_MD` below to change what the agent should do
(same role as `agent/instruction.md` in the harness).


In [ ]:
INSTRUCTION_MD = """# Ansible Playbook Builder Job

## Purpose

You are an Ansible automation engineer. You will be given playbook request forms as `.txt` files
in the inbox. Each request is written by an IT operations user who is not familiar with Ansible —
the request describes **what** needs to be done in plain language. Your job is to translate those
requirements into a correct, production-quality Ansible playbook.

Your job is to:
1. Read and parse each request file
2. Generate one Ansible playbook `.yml` file per request
3. Write a build summary report listing what was generated and any assumptions made

---

## Request File Format

Each `.txt` request file uses this structure:

```
REQUEST ID: <id>
DATE: <date>
REQUESTED BY: <name>

--- TARGET ---
HOSTNAME: <hostname or comma-separated list>
IP ADDRESS: <ip or comma-separated list>
OS: <Red Hat Enterprise Linux | Windows Server | Ubuntu | ...>
OS VERSION: <version>
BECOME: <yes | no>

--- TASK ---
TITLE: <short title>
EXPECTED OUTCOME:
  <free-text description of what must be achieved, one item per line>

--- CONSTRAINTS ---
  <operational rules — written in plain language, no Ansible terminology>

--- NOTES ---
  <additional context>
```

Fields may be absent if not relevant. Use sensible defaults when a field is missing.

---

## Tasks

### Step 1 — Parse each request file

For each `.txt` file in the inbox:
- Extract all structured fields
- Identify the OS family: `linux` (RHEL, Ubuntu, etc.) or `windows`
- Identify whether multiple hosts are listed
- Map each item in EXPECTED OUTCOME to one or more Ansible tasks
- Flag any ambiguities you resolve with an assumption

### Step 2 — Generate the Ansible playbook

Write one `.yml` file to `outbox/` per request, named from the Request ID in lowercase
(e.g. Request ID `PB-2026-001` → `pb-2026-001.yml`).

---

## Playbook Standards — Apply to Every Generated Playbook

### Header comment block
Every playbook must begin with:
```yaml
# ============================================================
# Playbook: <TITLE>
# Request ID: <REQUEST ID>
# Generated for: <HOSTNAME>
# OS: <OS> <OS VERSION>
# Requested by: <REQUESTED BY> on <DATE>
# ============================================================
```

### Idempotency
All playbooks must be safe to run more than once. Use modules in their idempotent form
(e.g. `state: present`, `enabled: true`). Where idempotency cannot be guaranteed, add a
`when:` condition or a check task before the action, and note the limitation in the summary.

### Secrets and passwords
- **Never hardcode passwords or secrets** in the playbook
- Reference them as Ansible Vault variables using the naming pattern `vault_<purpose>`
  (e.g. `vault_svc_monitor_password`)
- Use Ansible Jinja2 variable syntax to reference the vault variable in the playbook task
- Add a comment above any vault variable reference explaining where to set the value:
  ```yaml
  # Set this value in your Ansible Vault file before running
  password: "<<vault_svc_monitor_password>>"
  ```
  (Replace `<<vault_svc_monitor_password>>` with proper Ansible Jinja2 double-brace variable syntax when writing the playbook)

### Validation tasks
End each logical section with a task that confirms the expected outcome:
- On Linux: use `ansible.builtin.command` or `ansible.builtin.shell` with `register` + `ansible.builtin.debug`
- On Windows: use `ansible.windows.win_shell` with `register` + `ansible.builtin.debug`, or use
  `ansible.windows.win_service_info` / `ansible.windows.win_reg_stat` for structured checks

### Multi-host plays
If multiple hostnames are listed, target them with a group name derived from the request title
and add a comment block at the top of the playbook showing the expected inventory entries:
```yaml
# Expected inventory entries:
# [db_servers]
# db-prod-01 ansible_host=192.168.20.10
# db-prod-02 ansible_host=192.168.20.11
```

---

## Connection Defaults

Determine the connection method and Ansible user from the OS field in the request.
Do not expect the request to specify these — apply the defaults below automatically.

### Linux targets (RHEL, Ubuntu, and any non-Windows OS)
```yaml
ansible_connection: ssh
ansible_user: ansible
ansible_ssh_common_args: '-o StrictHostKeyChecking=no'
```
- Use `become: true` at the play level when the request specifies `BECOME: yes`
- Use `become_method: sudo` (default; no need to specify unless the request says otherwise)

### Windows targets (any Windows Server OS)
```yaml
ansible_connection: winrm
ansible_user: ansible_svc          # domain-joined: DOMAIN\ansible_svc
ansible_winrm_transport: kerberos  # use ntlm for workgroup (non-domain) hosts
ansible_winrm_server_cert_validation: ignore
ansible_port: 5985
```
- Set `become: false` at the play level — Windows privilege is controlled by the `ansible_user` credential
- If the hostname contains a domain hint (e.g. `CORP\...` style notes in the request), set transport to `kerberos`; otherwise default to `ntlm`

Include these connection variables in the play `vars:` block of every generated playbook so the
playbook is self-contained and does not depend on inventory-level variable files.

---

## Linux Playbook Rules

- Use `ansible.builtin.*` modules for standard tasks (always available, no install needed)
- Use `ansible.posix.*` for firewall, SELinux, mount, and authorized_key tasks
- Use `community.general.*` only when no builtin or posix module covers the task

### Firewall (Linux)
- Always use `ansible.posix.firewalld` — never use `ansible.builtin.shell` with `firewall-cmd`
- Always set both `permanent: true` and `immediate: true` so rules apply now and survive reboots
- Add a separate `ansible.posix.firewalld` task to ensure the service itself is enabled and running:
  ```yaml
  - name: Ensure firewalld is running and enabled
    ansible.builtin.systemd:
      name: firewalld
      state: started
      enabled: true
  ```

### User and group management (Linux)
- Use `ansible.builtin.group` to create the group before creating the user
- Use `ansible.builtin.user` for account creation; set `password: '!'` for accounts with no login password
- Use `ansible.posix.authorized_key` to deploy SSH public keys — never write to `~/.ssh/authorized_keys` directly
- Set `ansible.builtin.file` to enforce home directory permissions after user creation

### Sudo access (Linux)
- Use `community.general.sudoers` to create sudoers entries — never edit `/etc/sudoers` directly
- Always set `validate: true` in the module so the entry is checked with `visudo` before being written
- Scope each entry to specific commands; never grant `ALL` unless explicitly requested

### Service management (Linux)
- Use `ansible.builtin.systemd` with `daemon_reload: true` after any unit file change
- Always set both `state: started` and `enabled: true` unless the request says otherwise

---

## Windows Playbook Rules

- Use `ansible.windows.*` modules for all tasks
- Use `community.windows.*` only when no `ansible.windows` module covers the task
- Never use `ansible.builtin.command`, `ansible.builtin.shell`, or `ansible.builtin.copy` on Windows targets;
  use `ansible.windows.win_command`, `ansible.windows.win_shell`, and `ansible.windows.win_copy` instead

### User and group management (Windows)
- Use `ansible.windows.win_user` for local account creation
- Always set `update_password: on_create` so re-running the playbook never resets an existing account's password
- Use `ansible.windows.win_group_membership` to assign accounts to groups
- Reference passwords as Vault variables (see Secrets section above)

### Windows features and roles
- Use `ansible.windows.win_feature` to install Windows roles and features
- Set `include_management_tools: true` when installing server roles that have management tools

### IIS management
- Use `community.windows.win_iis_webapppool` for application pool creation and configuration
- Use `community.windows.win_iis_website` for website creation and binding configuration
- Always create and configure the application pool before creating the website that uses it
- Use `ansible.windows.win_copy` or `ansible.windows.win_template` to deploy web content files
- Use `ansible.windows.win_file` to create directory structures before copying content into them

### Firewall rules (Windows)
- Use `ansible.windows.win_firewall_rule` — never use `ansible.windows.win_shell` with `netsh`

### Registry changes (Windows)
- Use `ansible.windows.win_regedit` — never use `ansible.windows.win_shell` with `reg.exe` or `Set-ItemProperty`
- Always use PowerShell PSDrive syntax for paths: `HKLM:\...` not `HKEY_LOCAL_MACHINE\...`

### Service management (Windows)
- Use `ansible.windows.win_service` for all service state and startup type changes
- Use `ansible.windows.win_service_info` to verify service state in validation tasks

---

## Module Selection Reference

Prefer modules in this order:
1. `ansible.builtin.*` — always available
2. `ansible.windows.*` / `ansible.posix.*` — first-party, well-maintained
3. `community.windows.*` / `community.general.*` — community, widely used
4. `ansible.windows.win_shell` / `ansible.builtin.shell` — last resort when no module exists

| Task | Linux module | Windows module |
|------|-------------|----------------|
| Install package | `ansible.builtin.dnf` / `ansible.builtin.apt` | `ansible.windows.win_feature` |
| Manage service | `ansible.builtin.systemd` | `ansible.windows.win_service` |
| Copy file | `ansible.builtin.copy` | `ansible.windows.win_copy` |
| Template file | `ansible.builtin.template` | `ansible.windows.win_template` |
| Create directory | `ansible.builtin.file` | `ansible.windows.win_file` |
| Create user | `ansible.builtin.user` | `ansible.windows.win_user` |
| Create group | `ansible.builtin.group` | `ansible.windows.win_group` |
| Group membership | _(handled by `ansible.builtin.user`)_ | `ansible.windows.win_group_membership` |
| SSH authorized key | `ansible.posix.authorized_key` | _(N/A)_ |
| Sudoers entry | `community.general.sudoers` | _(N/A)_ |
| Firewall rule | `ansible.posix.firewalld` | `ansible.windows.win_firewall_rule` |
| Registry key | _(N/A)_ | `ansible.windows.win_regedit` |
| Read registry | _(N/A)_ | `ansible.windows.win_reg_stat` |
| IIS website | _(N/A)_ | `community.windows.win_iis_website` |
| IIS app pool | _(N/A)_ | `community.windows.win_iis_webapppool` |
| Run command | `ansible.builtin.command` | `ansible.windows.win_command` |
| Run script | `ansible.builtin.shell` | `ansible.windows.win_shell` |
| Wait for port | `ansible.builtin.wait_for` | `ansible.windows.win_wait_for` |
| Set file permissions | `ansible.builtin.file` | `ansible.windows.win_acl` |
| Cron / scheduled task | `ansible.builtin.cron` | `community.windows.win_scheduled_task` |
| Grant user right | `community.general.sudoers` | `ansible.windows.win_user_right` |
| Service info | `ansible.builtin.service_facts` | `ansible.windows.win_service_info` |

---

### Step 3 — Write the build summary

Write `build-summary.md` to `outbox/` with this structure per request:

```
## <REQUEST ID> — <TITLE>

**Input file**: <filename.txt>
**Output file**: <filename.yml>
**Target**: <hostname> (<OS> <VERSION>)
**Tasks generated**: <count>
**Modules used**: <comma-separated list>
**Assumptions made**:
  - <any field that was missing or ambiguous and how you resolved it>
**Warnings**:
  - <anything that could not be fully automated or needs human review before running>
```

---

## Output Files

For N request files in the inbox, write to `outbox/`:
- N playbook files named `<request-id-lowercase>.yml`
- One `build-summary.md` covering all requests
"""

print(f"Instruction: {len(INSTRUCTION_MD):,} characters")


## Step 3 — Input payloads

Edit the payload cells below. Each payload has a **filename** and **content**.
Leave a payload blank (`""`) to skip it — only non-empty payloads are loaded into `INPUT_FILES`.


In [ ]:
PAYLOAD_1_FILENAME = "pb-2026-002.txt"

PAYLOAD_1 = """REQUEST ID: PB-2026-002
DATE: 2026-06-20
REQUESTED BY: Marcus Webb

--- TARGET ---
HOSTNAME: db-prod-01, db-prod-02, db-prod-03
IP ADDRESS: 192.168.20.10, 192.168.20.11, 192.168.20.12
OS: Red Hat Enterprise Linux
OS VERSION: 8.9
BECOME: yes

--- TASK ---
TITLE: Create application service account and configure limited sudo access

EXPECTED OUTCOME:
  A system user account named "appuser" exists on all three hosts with:
    - UID: 1500
    - Primary group: appgroup (GID 1500)
    - Home directory: /opt/appuser
    - Shell: /bin/bash
    - No login password (SSH key authentication only)
  The following SSH public key is authorized for appuser:
    ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAABgQC7vX... appuser@controller
  appuser can run the following commands as root without a password prompt:
    /opt/app/bin/start.sh
    /opt/app/bin/stop.sh
  appuser cannot run any other commands as root
  The /opt/appuser directory is owned by appuser:appgroup and not accessible to other users

--- CONSTRAINTS ---
  All three hosts must be configured identically
  Running this automation more than once must not cause errors or duplicate entries

--- NOTES ---
These are the three PostgreSQL replica database servers.
The appuser account is needed for the application deployment scripts to run maintenance tasks.
"""


In [ ]:
PAYLOAD_2_FILENAME = ""

PAYLOAD_2 = """
"""


In [ ]:
PAYLOAD_3_FILENAME = ""

PAYLOAD_3 = """
"""


In [ ]:
INPUT_FILES: dict[str, str] = {}

for filename, content in (
    (PAYLOAD_1_FILENAME, PAYLOAD_1),
    (PAYLOAD_2_FILENAME, PAYLOAD_2),
    (PAYLOAD_3_FILENAME, PAYLOAD_3),
):
    name = (filename or "").strip()
    text = (content or "").strip()
    if name and text:
        INPUT_FILES[name] = text.rstrip() + "\n"

if not INPUT_FILES:
    raise ValueError("No payloads configured — fill in at least one PAYLOAD_N cell.")

for name, text in INPUT_FILES.items():
    print(f"  {name}: {len(text):,} chars")


## Step 4 — Prepare the workspace

Write payloads to disk so the file-based tools behave like `pydantic_agent.py`.


In [ ]:
import shutil

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

INPUT_DIR.mkdir(parents=True)
OUTPUT_DIR.mkdir(parents=True)

for name, content in INPUT_FILES.items():
    (INPUT_DIR / name).write_text(content, encoding="utf-8")

print(f"Input files : {sorted(p.name for p in INPUT_DIR.iterdir())}")
print(f"Output dir  : {OUTPUT_DIR.resolve()}")


## Step 5 — Tool implementations

Pydantic AI infers each tool's JSON schema from type annotations and docstrings.


In [ ]:
import json
import subprocess
from urllib.parse import urlparse

import httpx

WEB_FETCH_MAX_BYTES = 512_000
WEB_FETCH_DEFAULT_TIMEOUT = 30


def _safe_resolve(base: Path, rel: str) -> Path:
    target = (base / rel).resolve()
    if not str(target).startswith(str(base.resolve())):
        raise ValueError("Path traversal denied")
    return target


def _list_files(base: Path, exclude: set[str] | None = None) -> list[str]:
    exclude = exclude or set()
    if not base.exists():
        return []
    return sorted(
        str(p.relative_to(base)).replace("\\", "/")
        for p in base.rglob("*")
        if p.is_file() and p.name not in exclude
    )


def _validate_fetch_url(url: str) -> str:
    parsed = urlparse(url.strip())
    if parsed.scheme not in ("http", "https"):
        raise ValueError("URL must use http:// or https://")
    if not parsed.netloc:
        raise ValueError("URL must include a host")
    return url.strip()


def make_tools(input_dir: Path, output_dir: Path) -> list:
    input_r = input_dir.resolve()
    output_r = output_dir.resolve()

    def list_input_files() -> str:
        """List payload files in the input folder."""
        return json.dumps(_list_files(input_r))

    def read_input_file(path: str) -> str:
        """Read a file from the input folder."""
        try:
            target = _safe_resolve(input_r, path)
        except ValueError as exc:
            return f"Error: {exc}"
        if not target.is_file():
            return f"File not found: {path}"
        return target.read_text(encoding="utf-8", errors="replace")

    def write_output(path: str, content: str) -> str:
        """Write a file to the output folder."""
        try:
            target = _safe_resolve(output_r, path)
        except ValueError as exc:
            return f"Error: {exc}"
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return f"Wrote {target.stat().st_size} bytes to {path}"

    def append_output(path: str, content: str) -> str:
        """Append to a file in the output folder."""
        try:
            target = _safe_resolve(output_r, path)
        except ValueError as exc:
            return f"Error: {exc}"
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("a", encoding="utf-8") as fh:
            fh.write(content)
        return f"Appended {len(content)} chars to {path}"

    def list_output_files() -> str:
        """List files in the output folder."""
        return json.dumps(_list_files(output_r, exclude={"agent.log"}))

    def run_command(command: str, timeout: int = 60) -> str:
        """Execute a shell command in the output folder."""
        if not ALLOW_SHELL:
            return "Shell access is disabled (ALLOW_SHELL=false)"
        effective_timeout = min(int(timeout), SHELL_TIMEOUT)
        try:
            result = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                timeout=effective_timeout,
                cwd=str(output_r),
            )
            return json.dumps({
                "stdout": result.stdout,
                "stderr": result.stderr,
                "returncode": result.returncode,
            })
        except subprocess.TimeoutExpired:
            return json.dumps({"stdout": "", "stderr": "Command timed out", "returncode": -1})

    def web_fetch(url: str, timeout: int = WEB_FETCH_DEFAULT_TIMEOUT) -> str:
        """Fetch a URL over HTTP/HTTPS."""
        try:
            fetch_url = _validate_fetch_url(url)
        except ValueError as exc:
            return json.dumps({"error": str(exc)})
        effective_timeout = min(max(int(timeout), 1), 60)
        try:
            with httpx.Client(
                follow_redirects=True,
                timeout=effective_timeout,
                headers={"User-Agent": "pydantic-agent-tutorial/1.0"},
            ) as client:
                with client.stream("GET", fetch_url) as resp:
                    chunks: list[bytes] = []
                    size = 0
                    truncated = False
                    for chunk in resp.iter_bytes():
                        if size + len(chunk) > WEB_FETCH_MAX_BYTES:
                            remaining = WEB_FETCH_MAX_BYTES - size
                            if remaining > 0:
                                chunks.append(chunk[:remaining])
                            truncated = True
                            break
                        chunks.append(chunk)
                        size += len(chunk)
                    body = b"".join(chunks)
                    return json.dumps({
                        "url": str(resp.url),
                        "status_code": resp.status_code,
                        "content_type": resp.headers.get("content-type", ""),
                        "truncated": truncated,
                        "text": body.decode("utf-8", errors="replace"),
                    })
        except httpx.TimeoutException:
            return json.dumps({"error": f"Request timed out after {effective_timeout}s"})
        except httpx.HTTPError as exc:
            return json.dumps({"error": str(exc)})

    return [
        list_input_files, read_input_file, write_output, append_output,
        list_output_files, run_command, web_fetch,
    ]

tools = make_tools(INPUT_DIR, OUTPUT_DIR)
print(f"Registered {len(tools)} tools:", [t.__name__ for t in tools])


## Step 6 — Build the system prompt

In [ ]:
def build_system_prompt(instruction: str, skills_block: str = "") -> str:
    if ALLOW_SHELL:
        shell_block = (
            f"Shell guidance:\n"
            f"  - Prefer targeted commands (grep, awk, python, jq, curl, ansible-lint)\n"
            f"  - Working directory for commands is the output folder.\n"
            f"  - Commands time out after {SHELL_TIMEOUT} seconds.\n"
        )
    else:
        shell_block = "SHELL ACCESS DISABLED -- run_command will return an error.\n"

    prompt = (
        "You are a capable, autonomous AI agent running inside a secure container.\n"
        "You interact with the world ONLY through the tools listed below.\n"
        "\n"
        "Available tools:\n"
        "  - list_input_files  -- list payload files in the input folder\n"
        "  - read_input_file   -- read a file from the input folder\n"
        "  - write_output      -- write a file to the output folder\n"
        "  - append_output     -- append to a file in the output folder\n"
        "  - list_output_files -- list files already written to the output folder\n"
        "  - run_command       -- execute a shell command\n"
        "  - web_fetch         -- fetch an http(s) URL\n"
        "\n"
        "Large file guidance:\n"
        "  - Keep each write_output call under ~8 KB of content.\n"
        "  - For larger files, use append_output in several smaller chunks.\n"
        "\n"
        f"{shell_block}"
    )
    if skills_block:
        prompt += f"\n## Skills\n\n{skills_block}\n"
    prompt += f"\nTASK INSTRUCTIONS:\n{instruction}"
    return prompt


SYSTEM_PROMPT = build_system_prompt(INSTRUCTION_MD)
KICKOFF_PROMPT = "Begin executing the task instructions now."

print("System prompt preview (first 500 chars):")
print(SYSTEM_PROMPT[:500], "...")


## Step 7 — Build the model

Uses Pydantic AI's `OpenAIChatModel` pointed at `OPENAI_BASE_URL`.
Same code path for Ollama, vLLM, OpenRouter, or any OpenAI-compatible server.


In [ ]:
def build_model(model_id: str, *, max_tokens: int):
    from pydantic_ai.models.openai import OpenAIChatModel
    from pydantic_ai.providers.openai import OpenAIProvider

    return OpenAIChatModel(
        model_id,
        provider=OpenAIProvider(
            base_url=OPENAI_BASE_URL,
            api_key=OPENAI_API_KEY or "local",
        ),
    )


model = build_model(MODEL, max_tokens=MAX_OUTPUT_TOKENS)
print(f"Model ready: {MODEL} @ {OPENAI_BASE_URL}")


## Step 8 — Run the agent

Streams tool calls and assistant text to the cell output.


In [ ]:
import asyncio
from typing import Callable

from pydantic_ai import (
    Agent,
    AgentRunResultEvent,
    FunctionToolCallEvent,
    FunctionToolResultEvent,
    ModelSettings,
    PartDeltaEvent,
    TextPartDelta,
)


async def run_agent(*, log_callback: Callable[[str], None] | None = print) -> dict:
    agent = Agent(
        model=model,
        system_prompt=SYSTEM_PROMPT,
        tools=tools,
        model_settings=ModelSettings(max_tokens=MAX_OUTPUT_TOKENS),
    )

    turns = 0
    tokens_in = 0
    tokens_out = 0
    current_text: list[str] = []

    def emit(message: str) -> None:
        if log_callback:
            log_callback(message)

    emit(f"model={MODEL}  base_url={OPENAI_BASE_URL}  max_output_tokens={MAX_OUTPUT_TOKENS}")
    emit(f"input={INPUT_DIR}  output={OUTPUT_DIR}")

    async with agent.run_stream_events(KICKOFF_PROMPT) as events:
        async for event in events:
            if isinstance(event, PartDeltaEvent) and isinstance(event.delta, TextPartDelta):
                delta = event.delta.content_delta
                if delta:
                    current_text.append(delta)
                    combined = "".join(current_text)
                    if "\n" in combined:
                        lines = combined.split("\n")
                        for line in lines[:-1]:
                            if line.strip():
                                emit(f"[assistant] {line}")
                        current_text = [lines[-1]]

            elif isinstance(event, FunctionToolCallEvent):
                if current_text:
                    text = "".join(current_text).strip()
                    if text:
                        emit(f"[assistant] {text}")
                    current_text = []
                name = event.part.tool_name
                args = event.part.args or {}
                if isinstance(args, str):
                    args_str = args[:200]
                else:
                    args_str = ", ".join(
                        f"{k}={repr(str(v))[:80]}" for k, v in dict(args).items()
                    )
                emit(f"[tool_use] {name}({args_str})")
                turns += 1

            elif isinstance(event, FunctionToolResultEvent):
                tool_name = getattr(event.part, "tool_name", "tool")
                content = str(getattr(event.part, "content", ""))[:300]
                emit(f"[result] {tool_name}: {content}")

            elif isinstance(event, AgentRunResultEvent):
                usage = event.result.usage
                tokens_in = getattr(usage, "input_tokens", 0) or 0
                tokens_out = getattr(usage, "output_tokens", 0) or 0

    if current_text:
        text = "".join(current_text).strip()
        if text:
            emit(f"[assistant] {text}")

    emit(
        f"[result] turns={turns}  "
        f"tokens={tokens_in if tokens_in else '?'} in / "
        f"{tokens_out if tokens_out else '?'} out"
    )
    return {
        "total_turns": turns,
        "total_input_tokens": tokens_in,
        "total_output_tokens": tokens_out,
    }


# Jupyter / IPython 7+ supports top-level await:
stats = await run_agent()

# If top-level await is not available:
# stats = asyncio.run(run_agent())

stats


## Step 9 — Inspect output files

In [ ]:
from IPython.display import Markdown, display

output_files = sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file())

if not output_files:
    display(Markdown("*No output files yet — run Step 8 first.*"))
else:
    for path in output_files:
        rel = path.relative_to(OUTPUT_DIR)
        body = path.read_text(encoding="utf-8", errors="replace")
        display(Markdown(f"### `{rel}` ({path.stat().st_size:,} bytes)"))
        display(Markdown(f"```\n{body[:8000]}\n```"))


---

## Mapping to `pydantic_agent.py`

| Harness function | Notebook step |
|------------------|---------------|
| `load_agent_folder()` | Step 2 — `INSTRUCTION_MD` |
| Input folder | Steps 3–4 — `PAYLOAD_1`…`3` → `INPUT_FILES` |
| `_make_tools()` | Step 5 |
| `build_system_prompt()` | Step 6 |
| `_build_model()` | Step 7 — `OpenAIChatModel` + `OPENAI_BASE_URL` |
| `run_agent()` | Step 8 |

**Try next:**
- Edit `PAYLOAD_2` / `PAYLOAD_3` for multi-file jobs
- Point `OPENAI_BASE_URL` at OpenRouter or your local Ollama server
- Add custom tools (see `CUSTOM_TOOLS` in `pydantic_agent.py`)
